# Brasa ML on a Colab GPU: train, compare with the old U-Net, predict

Runs the whole `ml/` pipeline (see `ml/README.md`). Every step skips work whose output already exists,
so after a disconnect just *Run all* again.

| Step | What | Time (T4, rough) |
|---|---|---|
| 1 | Tensor stacks: overlap (train), eval (old + new channels), full (2000-2025) | ~30 min, first run only |
| 2 | *(optional)* ablation sweep: which change helps? | ~2 h |
| 3 | v2 ensemble on 2012-2018 (7 year folds) and **old vs new** on 2019-2025 | ~1.5 h |
| 4 | Backcast check: train 2016-2025, test 2012-2014 | ~30 min |
| 5 | Production model: same recipe on every overlap year 2012-2025 | ~1.5 h |
| 6 | Predict 2000-11 .. 2025-12, export for the app, merge, rebuild the globe | ~20 min |

**Before you start**: *Runtime -> Change runtime type -> T4 GPU* (or better). Put `brasa-colab-v2.tar`
in Google Drive at `MyDrive/brasa/`; for step 6 also the current app data in `MyDrive/brasa/app_m3/`.
Results go to `MyDrive/brasa/v2/`. Send back `report.md` files from `v2/eval_*`.

In [ ]:
# --- settings ---------------------------------------------------------------
BRASA = "/content/drive/MyDrive/brasa"
BUNDLE = f"{BRASA}/brasa-colab-v2.tar"
OUT = f"{BRASA}/v2"                  # persisted on Drive: checkpoints, reports, predictions
WORK = "/content/work"               # fast local disk (lost on disconnect, rebuilt automatically)
RUN_ABLATIONS = False                # step 2 costs ~2 GPU hours
FOLDS, SEEDS = 7, 1                  # ensemble size = FOLDS x SEEDS
APP_MODEL = "old"                    # model behind the app data (step 5): "old" = model/unet.pt, "new" = v2

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
from google.colab import drive
drive.mount("/content/drive")
import os, glob, shutil, json
for d in (OUT, WORK):
    os.makedirs(d, exist_ok=True)
if not os.path.exists(f"{WORK}/brasa/ml/run_train.py"):
    !tar -xf "{BUNDLE}" -C "{WORK}"
REPO, DER = f"{WORK}/brasa", f"{WORK}/firms/derived"
OLD = f"{REPO}/model/unet.pt"
%cd {REPO}
!pip -q install duckdb
!python -I ml/test_lib.py

## Step 1 - tensor stacks
`overlap` (v2 channels, 2011-12 .. 2025) trains every model; `eval` (all channels, 2018-2025) lets the old
and the new model be scored on one stack; `full` (v2, 2000-2025) is for prediction.

In [ ]:
def stack(name, years, preset):
    if not os.path.exists(f"{DER}/ml/{name}/meta.json"):
        !python -I ml/run_tensor.py {DER} --years {years} --preset {preset} --name {name}
    else:
        print(name, "already built")
stack("overlap", "2011-2025", "v2")
stack("eval", "2018-2025", "all")

## Step 2 - ablation sweep (optional)
Short runs, each scored against the old model on 2019-2024 at 5 degrees. `legacy-recipe` reproduces the
old training (combined channels, target-weighted sampling, 2-degree loss) as the control.

In [ ]:
# name: (stack, extra run_train.py flags). "legacy-recipe" is the control: the old training recipe.
ABLATIONS = {
    "legacy-recipe":  ("overlap_legacy", "--preset legacy --norm group --sampling target --reg-scales 8"),
    "v2":             ("overlap", ""),
    "v2-groupnorm":   ("overlap", "--norm group"),
    "v2-no-location": ("overlap", "--no-location"),
    "v2-depth2":      ("overlap", "--depth 2"),
    "v2-reg0":        ("overlap", "--reg-w 0"),
    "v2-reg1":        ("overlap", "--reg-w 1.0"),
    "v2-lag12":       ("overlap", "--offsets -12,-1,0,1"),
}
if RUN_ABLATIONS:
    import pandas as pd
    os.makedirs(f"{OUT}/ablations", exist_ok=True)
    stack("overlap_legacy", "2011-2025", "legacy")
    rows = []
    for name, (st, flags) in ABLATIONS.items():
        ck, ev = f"{OUT}/ablations/{name}.pt", f"{OUT}/ablations/eval_{name}"
        if not os.path.exists(ck):
            !python -I ml/run_train.py {DER}/ml/{st} {ck} --train 2012-2017 --val 2018 --epochs 20 {flags} 2>&1 | tail -2
        if not os.path.exists(f"{ev}/scores.csv"):
            !python -I ml/run_evaluate.py {DER} {DER}/ml/eval --model old={OLD} --model {name}={ck} --out {ev} --bootstrap 200 > /dev/null
        s = pd.read_csv(f"{ev}/scores.csv")
        rows.append(s[s.model == name].set_index("scale").med_abs_log_err.rename(name))
    print(pd.concat(rows, axis=1).T)   # median |log err| per scale; compare each row with "v2"

## Step 3 - v2 ensemble on 2012-2018, then old vs new on 2019-2025
Seven members, each early-stopped on its own held-out year (blocked CV), so no single validation year
decides. Also scored with Aqua removed (simulated Terra-only months) and with permutation importance.

In [ ]:
CV = f"{OUT}/unet_v2_cv2018.pt"
if not os.path.exists(CV):
    !python -I ml/run_train.py {DER}/ml/overlap {CV} --train 2012-2018 --folds {FOLDS} --seeds {SEEDS} 2>&1 | tee {OUT}/train_cv2018.log | grep -v epoch
EV = f"{OUT}/eval_2019-2025"
if not os.path.exists(f"{EV}/report.md"):
    !python -I ml/run_evaluate.py {DER} {DER}/ml/eval --model old={OLD} --model new={CV} --test 2019-2025 --terra-only --importance --out {EV} > {OUT}/eval_2019-2025.log
print(open(f"{EV}/report.md").read())

## Step 4 - backcast check (closest proxy for the 2000-2011 use)
Train on 2016-2025, test on 2012-2014. The old model saw those years in training, so the report flags it
as in-sample: compare the new model with Brasa M3 here, not with the old model.

In [ ]:
BC = f"{OUT}/unet_v2_backcast.pt"
if not os.path.exists(BC):
    !python -I ml/run_train.py {DER}/ml/overlap {BC} --train 2016-2025 --val 2015 2>&1 | tee {OUT}/train_backcast.log | grep -v epoch
EB = f"{OUT}/eval_backcast_2012-2014"
if not os.path.exists(f"{EB}/report.md"):
    !python -I ml/run_evaluate.py {DER} {DER}/ml/overlap --model new-backcast={BC} --test 2012-2014 --brasa-train 2016-2024 --out {EB} > {OUT}/eval_backcast.log
print(open(f"{EB}/report.md").read()[:4000])

## Step 5 - production model: the step-3 recipe on every overlap year
Same folds-and-early-stopping recipe on 2012-2025 (two years per fold). Its quality is the step-3 report;
there is no held-out VIIRS year left to score it on.

In [ ]:
PROD = f"{OUT}/unet_v2.pt"
if APP_MODEL == "new" and not os.path.exists(PROD):
    !python -I ml/run_train.py {DER}/ml/overlap {PROD} --train 2012-2025 --folds {FOLDS} --seeds {SEEDS} 2>&1 | tee {OUT}/train_prod.log | grep -v epoch
if APP_MODEL == "new":
    shutil.copy(PROD, f"{REPO}/model/unet_v2.pt")

## Step 6 - predict 2000-11 .. 2025-12, export for the app, merge, rebuild the globe
The v2 model covers the Terra-only months through its Aqua flag. `pipeline/08_merge_unet.py` decides
which cells and months stay on the ratio model (see its `--legacy` option).

In [ ]:
# The old model needs the legacy m_* channels and cannot do Terra-only months, so it starts at 2002-07.
APP_CK, APP_START, APP_STACK = (PROD, "2000-11", "full") if APP_MODEL == "new" else (OLD, "2002-07", "full_all")
stack(APP_STACK, "2000-2025", "v2" if APP_MODEL == "new" else "all")
PRED = f"{WORK}/pred_{APP_MODEL}"
if not os.path.exists(f"{PRED}/app_nn.parquet"):
    !python -I ml/run_predict.py {DER}/ml/{APP_STACK} {APP_CK} {PRED} --start {APP_START} --end 2025-12 --countries {DER}/app/cell_country.parquet --quantiles --app {DER}/ml/features.parquet
os.makedirs(f"{OUT}/pred", exist_ok=True)
for f in glob.glob(f"{PRED}/*"):
    shutil.copy(f, f"{OUT}/pred/")

In [ ]:
import duckdb
APP_IN = f"{BRASA}/app_m3"
D = f"{WORK}/merged"; A = f"{D}/app"
shutil.rmtree(D, ignore_errors=True); os.makedirs(A); os.makedirs(f"{D}/ml")
duckdb.sql(f"COPY (SELECT * FROM read_parquet('{APP_IN}/harm_part*.parquet') ORDER BY cell, year, month) "
           f"TO '{A}/harm.parquet' (FORMAT parquet, COMPRESSION zstd)")
for f in ("floor.parquet", "months.parquet", "cell_country.parquet", "model.json"):
    shutil.copy(f"{APP_IN}/{f}", A)
shutil.copy(f"{PRED}/app_nn.parquet", f"{D}/ml/app_nn.parquet")
!python -I {REPO}/pipeline/08_merge_unet.py {D}
!python -I {REPO}/pipeline/07_globe.py {D}
shutil.rmtree(f"{OUT}/app", ignore_errors=True)
shutil.copytree(A, f"{OUT}/app")
for f in sorted(glob.glob(f"{OUT}/app/*")):
    print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {os.path.basename(f)}")

## Quick look
`pred` is the v2 ensemble, `viirs_obs` observed S-NPP from 2012 on, `modis x 4.1` the naive global ratio.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
t = pd.read_csv(f"{PRED}/monthly_totals.csv")
g = t[t.region == "GLOBAL"].groupby("year")[["modis", "pred", "viirs_obs"]].sum(min_count=1)
g.insert(1, "modis x 4.1", 4.1 * g.modis)
print((g / 1e6).round(2).to_string())
ax = (g[["modis x 4.1", "pred", "viirs_obs"]] / 1e6).plot(marker="o", figsize=(10, 4))
ax.set_ylabel("detections per year (millions)"); ax.set_title("Global VIIRS-equivalent fire detections"); plt.show()